# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessica245818/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

My five contract answers, in plain words:

1. **One row means:** one pseudonymized content item at a **2026-03-15 decision moment**, after aggregating its daily observations.
2. **Table:** `fact_content_daily_performance`, using only the partition `month=2026-03`. This core contract does not need a join.
3. **Time window:** the five features use **2026-03-01 through 2026-03-15**; the decline proxy uses the non-overlapping outcome window **2026-03-16 through 2026-03-31**. June remains sealed as the final test month.
4. **What I rank:** content items by estimated probability of a later observed impressions decline, so an editor can choose which pages to review first for refresh, protection, metadata work, or monitoring.
5. **Deliberate exclusion:** post-decision impressions and anything derived from them are never model features. I add one such column only inside the leakage demonstration and then delete it. Pseudonymized IDs are context for grouping and validation, not features.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd

# Token order: environment -> Colab Secret -> private prompt. The token is never printed.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")
del safe_token

MARCH = (
    "read_parquet('hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet')"
)
print("Connected to the gated March 2026 warehouse partition.")


Note: you may need to restart the kernel to use updated packages.


Connected to the gated March 2026 warehouse partition.


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Contract |
|---|---|---|
| **Features (exactly five)** | `impressions_pre`, `clicks_pre`, `ctr_pre`, `avg_position_pre`, `active_days_pre` | Aggregated only from March 1–15 and knowable at the decision moment. |
| **Label / proxy** | `declined_post` | 1 when average daily impressions in March 16–31 are more than 20% below March 1–15. It is a later observed proxy for review opportunity, not proof a refresh would help. |
| **Label helper** | `impressions_post` | Builds the proxy from the future window; never passed to a model. |
| **Context** | `client_hash_id`, `content_hash_id` | Used for grain, joins, grouped validation, and readable output; IDs are not features. |
| **Availability/filter** | `report_date`, `gsc_data_available` | Defines the windows and keeps only rows where GSC data is truly available. |
| **Excluded** | all post-decision metrics, `ga4_*`, `sessions_*`, AI fields, raw/private fields | Future metrics leak the answer. GA4 and AI signals are outside this five-feature GSC contract. Raw names, URLs, queries, and titles are not in the safe release and must never be reconstructed. |

The output is a ranked decision-support queue for an editor, with scores and observable reason codes. A high score means “review first,” never “an edit is guaranteed to recover traffic.”

In [2]:
feature_cols = [
    "impressions_pre",
    "clicks_pre",
    "ctr_pre",
    "avg_position_pre",
    "active_days_pre",
]
available_when = {
    "impressions_pre": "knowable at the decision moment because it sums GSC impressions through March 15 only",
    "clicks_pre": "knowable at the decision moment because it sums GSC clicks through March 15 only",
    "ctr_pre": "knowable at the decision moment because it uses only pre-decision clicks and impressions",
    "avg_position_pre": "knowable at the decision moment because it averages observed positions through March 15 only",
    "active_days_pre": "knowable at the decision moment because it counts pre-decision days with impressions",
}

assert len(feature_cols) == 5
for feature in feature_cols:
    print(f"- {feature}: {available_when[feature]}.")


- impressions_pre: knowable at the decision moment because it sums GSC impressions through March 15 only.
- clicks_pre: knowable at the decision moment because it sums GSC clicks through March 15 only.
- ctr_pre: knowable at the decision moment because it uses only pre-decision clicks and impressions.
- avg_position_pre: knowable at the decision moment because it averages observed positions through March 15 only.
- active_days_pre: knowable at the decision moment because it counts pre-decision days with impressions.


## 3. Verify it with exactly three queries, then build features and spring the trap

The three verification queries below prove: (1) the daily grain has no duplicates, (2) the GSC-available slice's row count and date span match March, and (3) availability using literal `IS TRUE` filters. After those three outputs, a separate feature-building query aggregates the same partition into the five-feature decision frame.

**Feature availability:**

- `impressions_pre` — knowable at the decision moment because it sums impressions only through March 15.
- `clicks_pre` — knowable because it sums clicks only through March 15.
- `ctr_pre` — knowable because both its numerator and denominator stop at March 15.
- `avg_position_pre` — knowable because it averages only positions already observed by March 15.
- `active_days_pre` — knowable because it counts only pre-decision days with impressions.

In [3]:
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Verification query 1 of 3: prove the documented daily grain.
grain_query = f"""
SELECT COUNT(*) AS duplicate_grain_groups
FROM (
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_grain
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
)
"""
print("Verification query 1/3 — grain (zero duplicate groups means it holds)")
display(con.sql(grain_query).df())

# Verification query 2 of 3: row count, item count, and date span for the lane slice.
count_window_query = f"""
SELECT COUNT(*) AS slice_rows,
       COUNT(DISTINCT content_hash_id) AS content_items,
       MIN(report_date) AS min_date,
       MAX(report_date) AS max_date
FROM {MARCH}
WHERE gsc_data_available IS TRUE
"""
print("Verification query 2/3 — GSC-available slice count and window")
display(con.sql(count_window_query).df())

# Verification query 3 of 3: availability, explicitly filtered with IS TRUE.
availability_query = f"""
SELECT COUNT(*) AS rows_with_gsc_and_ga4
FROM {MARCH}
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
"""
print("Verification query 3/3 — rows surviving GSC IS TRUE and GA4 IS TRUE")
display(con.sql(availability_query).df())

# Feature-building SQL (not a fourth contract-verification query).
feature_sql = f"""
WITH per_content AS (
    SELECT client_hash_id,
           content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_pre,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_pre,
           AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0
                    THEN gsc_avg_position END) AS avg_position_pre,
           COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0
                               THEN report_date END) AS active_days_pre,
           SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_post
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT client_hash_id,
       content_hash_id,
       impressions_pre,
       clicks_pre,
       CASE WHEN impressions_pre > 0
            THEN 100.0 * clicks_pre / impressions_pre ELSE 0 END AS ctr_pre,
       avg_position_pre,
       active_days_pre,
       impressions_post,
       CAST((impressions_post / 16.0) < 0.8 * (impressions_pre / 15.0) AS INTEGER) AS declined_post
FROM per_content
WHERE impressions_pre >= 100
"""
feature_frame = con.sql(feature_sql).df().dropna(subset=feature_cols + ["declined_post"])
assert feature_frame["content_hash_id"].is_unique
print(f"Five-feature frame: {len(feature_frame):,} content items x {len(feature_cols)} features")
display(feature_frame[["content_hash_id"] + feature_cols + ["declined_post"]].head())

# Honest grouped split: whole clients stay on one side.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(
    splitter.split(feature_frame, groups=feature_frame["client_hash_id"])
)
honest_model = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=50, random_state=42
).fit(feature_frame.iloc[train_idx][feature_cols], feature_frame.iloc[train_idx]["declined_post"])
honest_score = roc_auc_score(
    feature_frame.iloc[test_idx]["declined_post"],
    honest_model.predict_proba(feature_frame.iloc[test_idx][feature_cols])[:, 1],
)

# DELIBERATE LEAK: this ratio uses post-decision impressions, which define the label.
feature_frame["future_impressions_ratio"] = (
    (feature_frame["impressions_post"] / 16.0)
    / (feature_frame["impressions_pre"] / 15.0)
)
leaky_cols = feature_cols + ["future_impressions_ratio"]
leaky_model = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=50, random_state=42
).fit(feature_frame.iloc[train_idx][leaky_cols], feature_frame.iloc[train_idx]["declined_post"])
leaky_score = roc_auc_score(
    feature_frame.iloc[test_idx]["declined_post"],
    leaky_model.predict_proba(feature_frame.iloc[test_idx][leaky_cols])[:, 1],
)

print(f"Honest held-out ROC-AUC (five pre-decision features): {honest_score:.3f}")
print(f"Leaky held-out ROC-AUC (+ future ratio):             {leaky_score:.3f}")

# Remove the trap and keep the honest result.
feature_frame.drop(columns=["future_impressions_ratio"], inplace=True)
del leaky_model, leaky_cols
assert "future_impressions_ratio" not in feature_frame.columns
assert len(feature_cols) == 5
print("Leak removed. Kept honest ROC-AUC:", f"{honest_score:.3f}")


Verification query 1/3 — grain (zero duplicate groups means it holds)


,duplicate_grain_groups
0,0


Verification query 2/3 — GSC-available slice count and window


,slice_rows,content_items,min_date,max_date
0,3611061,176738,2026-03-01,2026-03-31


Verification query 3/3 — rows surviving GSC IS TRUE and GA4 IS TRUE


,rows_with_gsc_and_ga4
0,364347


Five-feature frame: 77,540 content items x 5 features


,content_hash_id,impressions_pre,clicks_pre,ctr_pre,avg_position_pre,active_days_pre,declined_post
0,content_d35900ed79f98a43,312.0,2.0,0.641026,9.440840,15,0
1,content_abc23f421b72bcc9,1169.0,3.0,0.256630,18.904041,15,0
2,content_7302a89449110b7c,123.0,0.0,0.000000,18.151703,15,0
3,content_f22381b2e47510f1,136.0,1.0,0.735294,6.623279,14,1
4,content_77776527c2977241,992.0,3.0,0.302419,5.016898,15,0


Honest held-out ROC-AUC (five pre-decision features): 0.540
Leaky held-out ROC-AUC (+ future ratio):             1.000
Leak removed. Kept honest ROC-AUC: 0.540


## 4. Data limits

**Named limitation — single-month, unbalanced-history slice.** March contains only the clients and content active and tracked in that month; client history starts at different dates, and the availability query shows that far fewer rows have both GSC and GA4 than have GSC. This five-feature contract deliberately uses GSC only, so it cannot measure engagement quality, seasonality, or whether an edit caused recovery. The later outcome is an observed decline proxy—not “worth a refresh” ground truth. A capstone result must repeat the contract across earlier months, validate on unseen clients and later time, and keep June sealed until the final test.

In [4]:
print("Content items in feature frame:", f"{len(feature_frame):,}")
print("Pseudonymized clients represented:", feature_frame["client_hash_id"].nunique())
print("Target positive rate:", f"{feature_frame['declined_post'].mean():.1%}")
print("Final feature count:", len(feature_cols))
print("Leakage column present after cleanup:", "future_impressions_ratio" in feature_frame.columns)


Content items in feature frame: 77,540
Pseudonymized clients represented: 38
Target positive rate: 32.8%
Final feature count: 5
Leakage column present after cleanup: False


## 5. Self-check

- [x] Five plain-words contract answers are present.
- [x] Exactly three verification queries have visible outputs; availability uses `IS TRUE`.
- [x] The frame contains exactly five features, each with an “available when?” explanation.
- [x] One deliberate label-derived feature is demonstrated, deleted, and the honest score retained.
- [x] One named limitation is documented.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, private queries, or tokens appear anywhere.
- [x] Claims remain observed, measured, directional, and decision-support.
- [x] Saved under `work/notebooks/`; commit verification is completed with submission.